

Goal: Test correlations between measures

In [1]:
# libraries

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import numpy as np
from scipy.stats import zscore
import pingouin as pg


Prepare lists.

In [2]:
targets = ['P03_joy', 'P29_joy', 'P34_joy', 'P42_joy', 'P44_joy',
           'P12_sadness', 'P15_sadness', 'P27_sadness', 'P34_sadness', 'P44_sadness', 
           'P03_anger', 'P15_anger', 'P22_anger', 'P38_anger', 'P42_anger',
           'P17_pride', 'P22_pride', 'P29_pride', 'P42_pride', 'P44_pride',
           'P03_shame', 'P22_shame', 'P37_shame', 'P38_shame', 'P45_shame', 
           'P12_neutral', 'P17_neutral', 'P29_neutral', 'P34_neutral', 'P38_neutral']

perceivers = ["S02", "S03", "S04", "S05", "S06", "S07", "S08", "S11", "S15", "S20", "S23",
                "S24", "S28", "S34", "S37", "S42", "S44", "S45", "S47", "S49", "S51", "S54", "S55", "S57", "S64", "S65"]

# emotions on the Geneva Emotion Wheel (for specific emotion selection)
geneva_emotions = ['geneva_interest', 'geneva_amusement',	'geneva_pride',	'geneva_joy', 'geneva_pleasure',
                   'geneva_contentment',	'geneva_love',	'geneva_admiration',	'geneva_relief',	'geneva_compassion',
                   'geneva_sadness',	'geneva_guilt',	'geneva_regret',	'geneva_shame',	'geneva_disappointment',
                   'geneva_fear',	'geneva_disgust',	'geneva_contempt',	'geneva_hate',	'geneva_anger']

# data from previous script (EA scores; correspondance between each target and perceiver data pair)
correlation_data = pd.read_csv("outputs/correlation_EA_scores.csv") 
geneva_data = pd.read_csv("outputs/geneva_EA_scores.csv")
HR_data = pd.read_csv("outputs/BPM_synchrony_scores.csv")
HRV_data = pd.read_csv("outputs/RMSSD_synchrony_scores.csv")

Correlations between EA measures (correlation, geneva emotions, HR, HRV)

In [ ]:


plt.rcParams["font.family"] = "Times New Roman"

common_cols = ['perceiver', 'evaluation_of', 'compare_with_target', 'match_type', 'emotion'] # common columns between measures
data = correlation_data[common_cols + ['z_corr']].rename(columns = {'z_corr': 'raw_corr'})
data = data.merge(geneva_data[common_cols + ['sum_distance']].rename(columns = {'sum_distance': 'raw_gew'}), on=common_cols)
data = data.merge(HR_data[common_cols + ['z_corr']].rename(columns = {'z_corr': 'raw_HR'}), on=common_cols)
data = data.merge(HRV_data[common_cols + ['z_corr']].rename(columns = {'z_corr': 'raw_HRV'}), on=common_cols)

# filter 
data = data[data['match_type'] == 'match'].dropna() # ensure we only take into consideration the 'real' target-observer pairs 

# z-score each EA measure within each target

data['tz_corr'] = data.groupby('evaluation_of')['raw_corr'].transform(zscore)
data['tz_gew'] = data.groupby('evaluation_of')['raw_gew'].transform(zscore)
data['tz_HR'] = data.groupby('evaluation_of')['raw_HR'].transform(zscore)
data['tz_HRV'] = data.groupby('evaluation_of')['raw_HRV'].transform(zscore)
    
# aggregate: 1 score per perceiver per measure (average z-scores across trials)
grouped = data.groupby('perceiver')[['tz_corr', 'tz_gew', 'tz_HR', 'tz_HRV']].mean().reset_index()

variables = grouped[['tz_corr', 'tz_gew', 'tz_HR', 'tz_HRV']]

labels = ['CV-EA', 'SE-EA', 'HR-S', 'HRV-S'] 

# plot for correlations
fig, axs = plt.subplots(4, 4, figsize=(10, 10))

for i, ylab in enumerate(labels):
    for j, xlab in enumerate(labels):
        ax = axs[i, j]
        x = variables.iloc[:, j] # all variables
        y = variables.iloc[:, i] # all variables

        ax.scatter(x, y, s=5, color='black')
        r, p = spearmanr(x, y)
        ax.plot(np.sort(x), np.poly1d(np.polyfit(x, y, 1))(np.sort(x)), color='red')
        
        # add r and p 
        ax.annotate(f"r = {r:.2f}\np = {p:.2f}", 
                    xy=(0.04, 0.20), 
                    xycoords='axes fraction',
                    fontsize=9, ha='left', va='top',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none'))

        if i == len(labels) - 1:
            ax.set_xlabel(xlab, fontsize=10)
        if j == 0:
            ax.set_ylabel(ylab, fontsize=10)

plt.tight_layout()
plt.show()


split-half correlations to assess interindividual consistency

In [ ]:

def splithalf(data, score_column, n_repeats=1000):
    
    raw_corrs = [] # store raw Spearman r
    corrected_corrs = [] # store Spearman-Brown corrected r

    for i in range(n_repeats): # for each repeat
        
        a_vals = []  # mean scores from first half (for each perceiver)
        b_vals = []  # mean scores from second half (for each perceiver)

        for perceiver, group in data.groupby('perceiver'): # loop through each perceiver
            
            values = group[score_column].dropna().values
            np.random.shuffle(values) # shuffle values
            half = len(values) // 2
            a = values[:half] # take first half
            b = values[half:] # take second half
            a_vals.append(np.mean(a)) # take mean of first half
            b_vals.append(np.mean(b)) # take mean of second half

        r, p = spearmanr(a_vals, b_vals) # correlation across perceivers between A and B
        raw_corrs.append(r) # keep the r value

        corrected_r = (2 * r) / (1 + r) # Spearman-Brown correlation
        corrected_corrs.append(corrected_r) # keep corrected r
        
    return np.mean(raw_corrs), np.mean(corrected_corrs) # mean correlation

# results

r_raw, r_corrected = splithalf(data, score_column='tz_corr')
print(f"Correlation EA: split-half r = {round(r_raw, 4)}")

r_raw, r_corrected = splithalf(data, score_column='tz_gew')
print(f"GEW EA: split-half r = {round(r_raw, 4)}")

r_raw, r_corrected = splithalf(data, score_column='tz_HR')
print(f"HR: split-half r = {round(r_raw, 4)} ")

r_raw, r_corrected = splithalf(data, score_column='tz_HRV')
print(f"HRV: split-half r = {round(r_raw, 4)} ")



Convergent validity: correlations with validated measures of empathy

In [ ]:

# 1 score per perceiver for empathic accuracy and physiological synchrony measures

grouped_raw = data.groupby('perceiver').agg({
    'raw_corr': 'mean',
    'raw_gew': 'mean',
    'raw_HR': 'mean',
    'raw_HRV': 'mean'}).reset_index()

grouped_tz = data.groupby('perceiver').agg({
    'tz_corr': 'mean',
    'tz_gew': 'mean',
    'tz_HR': 'mean',
    'tz_HRV': 'mean'}).reset_index()

# import BES-A and MET data
questionnaires_data = pd.read_csv("behavioral_data/PERCEIVER_data_questionnaires.csv")

# BESA subscales
besa_ed_cols = [f"BESA_ED_{i}" for i in range(1, 7)] # emotional disconnection
besa_ce_cols = [f"BESA_CE_{i}" for i in range(1, 9)] # cognitive empathy
besa_ec_cols = [f"BESA_EC_{i}" for i in range(1, 7)] # emotional contagion
questionnaires_data['BESA_CE'] = questionnaires_data[besa_ce_cols].mean(axis=1, skipna=True)
questionnaires_data['BESA_EC'] = questionnaires_data[besa_ec_cols].mean(axis=1, skipna=True)
questionnaires_data['BESA_ED'] = questionnaires_data[besa_ed_cols].mean(axis=1, skipna=True)

# MET subscales
met_ce_cols = [f"MET_part1_{i}" for i in range(1, 41)] # cognitive empathy
met_ec_cols = [f"MET_part2_{i}" for i in range(1, 41)] # emotional contagion
questionnaires_data['MET_CE'] = questionnaires_data[met_ce_cols].mean(axis=1, skipna=True)
questionnaires_data['MET_EC'] = questionnaires_data[met_ec_cols].mean(axis=1, skipna=True)

questionnaires_data.to_excel("questionnaires_data_update.xlsx", index=False)

# merge datasets
merge_raw = pd.merge(
    grouped_raw[['perceiver', 'raw_corr', 'raw_gew', 'raw_HR', 'raw_HRV']],
    questionnaires_data[['perceiver', 'BESA_CE', 'BESA_EC', 'BESA_ED', 'MET_CE', 'MET_EC']],
    on=['perceiver'], how='inner')
merge_raw = merge_raw.drop(columns=['perceiver']) # keep just columns to analyze

merge_tz = pd.merge(
    grouped_tz[['perceiver', 'tz_corr', 'tz_gew', 'tz_HR', 'tz_HRV']],
    questionnaires_data[['perceiver', 'BESA_CE', 'BESA_EC', 'BESA_ED', 'MET_CE', 'MET_EC']],
    on=['perceiver'], how='inner')
merge_tz = merge_tz.drop(columns=['perceiver']) # keep just columns to analyze

# identify labels
labels = {
    'raw_corr': 'EA Correlation',
    'raw_gew': 'EA GEW',
    'raw_HR': 'BPM synchrony',
    'raw_HRV': 'RMSSD synchrony',
    'tz_corr': 'EA Correlation (z)',
    'tz_gew': 'EA GEW (z)',
    'tz_HR': 'BPM synchrony (z)',
    'tz_HRV': 'RMSSD synchrony (z)',
    'BESA_CE': 'BESA Cognitive',
    'BESA_EC': 'BESA Contagion',
    'BESA_ED': 'BESA Disconnection',
    'MET_CE': 'MET Cognitive',
    'MET_EC': 'MET Emotional'}

### correlation matrix

cols = merge_raw.columns
n = len(cols)
corr_matrix = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)
p_matrix = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)

for item1 in range(n):
    for item2 in range(n):
        r, p = spearmanr(merge_raw[cols[item1]], merge_raw[cols[item2]])
        corr_matrix.iloc[item1, item2] = r
        p_matrix.iloc[item1, item2] = p

print(f'--- correlation matrix (RAW) --- \n{corr_matrix}')
print(f'\n--- p value matrix (RAW) --- \n{p_matrix}')

corr_matrix = pd.DataFrame(corr_matrix)
corr_matrix.to_excel("outputs/corr_matrix_raw.xlsx", index=False)

p_matrix = pd.DataFrame(p_matrix)
p_matrix.to_excel("outputs/p_matrix_raw.xlsx", index=False)



## same with z-scores

cols = merge_tz.columns
n = len(cols)
corr_matrix = pd.DataFrame(np.zeros((n, n)), index=cols, columns=cols)
p_matrix = pd.DataFrame(np.ones((n, n)), index=cols, columns=cols)

for item1 in range(n):
    for item2 in range(n):
        r, p = spearmanr(merge_tz[cols[item1]], merge_tz[cols[item2]])
        corr_matrix.iloc[item1, item2] = r
        p_matrix.iloc[item1, item2] = p

print(f'--- correlation matrix (Z)--- \n{corr_matrix}')
print(f'\n--- p value matrix (Z) --- \n{p_matrix}')

corr_matrix = pd.DataFrame(corr_matrix)
corr_matrix.to_excel("outputs/corr_matrix_tz.xlsx", index=False)

p_matrix = pd.DataFrame(p_matrix)
p_matrix.to_excel("outputs/p_matrix_tz.xlsx", index=False)



Bayesian correlations

In [ ]:
variable_names = {'tz_corr': 'CV-EA', 'tz_gew': 'SE-EA', 'tz_HR': 'HR-S', 'tz_HRV': 'HRV-S',
         'BESA_CE': 'BESA Cognitive', 'BESA_EC': 'BESA Contagion', 'BESA_ED': 'BESA Disconnection',
         'MET_CE': 'MET Cognitive', 'MET_EC': 'MET Emotional'}

columns = list(variable_names)

bf10 = pd.DataFrame(np.nan, index=columns, columns=columns)

for column_a in columns:
    for column_b in columns:
        if column_a == column_b:
            continue
        pair = merge_tz[[column_a, column_b]].dropna() # check each pair              
        rho, _ = spearmanr(pair[column_a], pair[column_b]) # frequentist correlation
        bf10.loc[column_a, column_b] = pg.bayesfactor_pearson(rho, len(pair)) # rho + nb of pairs

print(bf10.round(2))